In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

prac-1

In [0]:



spark = SparkSession.builder.getOrCreate()

# Schema
schema = StructType([
    StructField("Club_Id", IntegerType(), True),
    StructField("Member_Id", IntegerType(), True),
    StructField("EDU", StringType(), True),
])

# Data
data = [
    (1001, 210, None),
    (1001, 211, "MM:CI"),
    (1002, 215, "CD:CI:CM"),
    (1002, 216, "CL:CM"),
    (1002, 217, "MM:CM"),
    (1003, 255, None),
    (1001, 216, "CO:CD:CL:MM"),
    (1002, 210, None),
]

df = spark.createDataFrame(data, schema)
df.show(truncate=False)



+-------+---------+-----------+
|Club_Id|Member_Id|EDU        |
+-------+---------+-----------+
|1001   |210      |NULL       |
|1001   |211      |MM:CI      |
|1002   |215      |CD:CI:CM   |
|1002   |216      |CL:CM      |
|1002   |217      |MM:CM      |
|1003   |255      |NULL       |
|1001   |216      |CO:CD:CL:MM|
|1002   |210      |NULL       |
+-------+---------+-----------+



In [0]:
df1=df.withColumn("EDU", split(col("EDU"), ":"))\
  .withColumn("EDU_code",explode_outer(col("EDU")))\
  .withColumn("edu_flag", when(col("EDU_code").isin("MM","CI","CO"),0.5 )
              .when(col("EDU_code").isin("CD","CL","CM"),1.0)
              .otherwise(0)
              )

In [0]:
df1.groupBy("Club_Id").agg(sum(col("edu_flag")).alias("flag")).show()
min_max=df1.select(min("Club_Id").alias("min"),max("Club_Id").alias("max")).collect()[0]
print(min_max["min"],min_max["max"])
print(df.columns)
new_column_names = [col.replace('_', '') for col in df1.columns]
df_renamed = df1.toDF(*new_column_names)
df_renamed.show(truncate=False)


+-------+----+
|Club_Id|flag|
+-------+----+
|   1001| 4.0|
|   1002| 6.0|
|   1003| 0.0|
+-------+----+

1001 1003
['Club_Id', 'Member_Id', 'EDU']
+------+--------+----------------+-------+-------+
|ClubId|MemberId|EDU             |EDUcode|eduflag|
+------+--------+----------------+-------+-------+
|1001  |210     |NULL            |NULL   |0.0    |
|1001  |211     |[MM, CI]        |MM     |0.5    |
|1001  |211     |[MM, CI]        |CI     |0.5    |
|1002  |215     |[CD, CI, CM]    |CD     |1.0    |
|1002  |215     |[CD, CI, CM]    |CI     |0.5    |
|1002  |215     |[CD, CI, CM]    |CM     |1.0    |
|1002  |216     |[CL, CM]        |CL     |1.0    |
|1002  |216     |[CL, CM]        |CM     |1.0    |
|1002  |217     |[MM, CM]        |MM     |0.5    |
|1002  |217     |[MM, CM]        |CM     |1.0    |
|1003  |255     |NULL            |NULL   |0.0    |
|1001  |216     |[CO, CD, CL, MM]|CO     |0.5    |
|1001  |216     |[CO, CD, CL, MM]|CD     |1.0    |
|1001  |216     |[CO, CD, CL, MM]|CL

In [0]:

pk_keys='club_id,member_id,edu'
pk_keys=pk_keys.split(',')
invalid_condition = (col("flag") > 100)
for key in pk_keys:
    invalid_condition = invalid_condition | col(key).isNull()

print(df1.schema.fields)
print(df.schema.names)


for field in df.schema.fields:
    col_name = field.name
    col_type = field.dataType.simpleString()   # or: str(field.dataType)

    print(col_name, col_type)


[StructField('Club_Id', IntegerType(), True), StructField('Member_Id', IntegerType(), True), StructField('EDU', ArrayType(StringType(), False), True), StructField('EDU_code', StringType(), True), StructField('edu_flag', DoubleType(), False)]
['Club_Id', 'Member_Id', 'EDU']
Club_Id int
Member_Id int
EDU string


In [0]:
df_fill = df.fillna("UNKNOWN")
df_fill = df.fillna({"EDU": "UNKNOWN", "Member_Id": 0})

df_fill = df.fillna("N/A", subset=["EDU"])
df_drop = df.dropna()
df_drop = df.dropna(how="all", subset=["EDU"])
df_drop = df.dropna(how="any", subset=["EDU"])

In [0]:


# Initialize Spark
# spark = SparkSession.builder.appName("PhoneNumbersExample").getOrCreate()

# Define schema
schema = StructType([
    StructField("num", StringType(), True)
])

# Data
data = [
    ('1234567780'),
    ('2234578996'),
    ('+1-12244567780'),
    ('+32-2233567889'),
    ('+2-23456987312'),
    ('+91-9087654123'),
    ('+23-9085761324'),
    ('+11-8091013345')
]

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Show DataFrame
df.show(truncate=False)
df.printSchema()


+--------------+
|num           |
+--------------+
|1234567780    |
|2234578996    |
|+1-12244567780|
|+32-2233567889|
|+2-23456987312|
|+91-9087654123|
|+23-9085761324|
|+11-8091013345|
+--------------+

root
 |-- num: string (nullable = true)



In [0]:
df_position = df.withColumn(
    "at_position",
    instr(col("num"), "-")
)
df_position.show()

df_mob = df.withColumn(
    "mob",
    get(split(col("num"), "-"),0)
).show()




# UDF to check unique digits
def has_unique_digits(phone):
    digits = ''.join(ch for ch in phone if ch.isdigit())  # Remove non-digits
    return len(digits) == len(set(digits))

unique_digits_udf = udf(has_unique_digits, BooleanType())


df1=df.withColumn("new_num",when( instr(col("num"),'-')==0,col('num')).otherwise(get(split(col("num"),"-"),1)))
df1.show()

# Apply UDF
df_filtered = df1.filter(unique_digits_udf(col("new_num")))

df_filtered.show(truncate=False)



+--------------+-----------+
|           num|at_position|
+--------------+-----------+
|    1234567780|          0|
|    2234578996|          0|
|+1-12244567780|          3|
|+32-2233567889|          4|
|+2-23456987312|          3|
|+91-9087654123|          4|
|+23-9085761324|          4|
|+11-8091013345|          4|
+--------------+-----------+

+--------------+----------+
|           num|       mob|
+--------------+----------+
|    1234567780|1234567780|
|    2234578996|2234578996|
|+1-12244567780|        +1|
|+32-2233567889|       +32|
|+2-23456987312|        +2|
|+91-9087654123|       +91|
|+23-9085761324|       +23|
|+11-8091013345|       +11|
+--------------+----------+

+--------------+-----------+
|           num|    new_num|
+--------------+-----------+
|    1234567780| 1234567780|
|    2234578996| 2234578996|
|+1-12244567780|12244567780|
|+32-2233567889| 2233567889|
|+2-23456987312|23456987312|
|+91-9087654123| 9087654123|
|+23-9085761324| 9085761324|
|+11-8091013345| 8091013

In [0]:

# Step 1: Get min and max
min_val = numbers.agg(min("n")).collect()[0][0]
max_val = numbers.agg({"n": "max"}).collect()[0][0]

# Step 2: Generate full range DataFrame
full_range_df = spark.range(min_val, max_val + 1).toDF("num")

# Step 3: Find missing numbers using left anti join
missing_df = full_range_df.join(numbers_df, full_range_df.num == numbers_df.n, "left_anti")

missing_show()


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6205127652660416>, line 3
      2 # Step 1: Get min and max
----> 3 min_val = numbers.agg(min("n")).collect()[0][0]
      4 max_val = numbers.agg({"n": "max"}).collect()[0][0]
      6 # Step 2: Generate full range DataFrame

NameError: name 'numbers' is not defined

In [0]:
df_position.withColumn("mob",substring(col("num"),instr(col("num"),"-")+1,length(col("num")))).show() 
df_position.withColumn("code",    regexp_extract(col("num"), r"(\+\d+)-", 1)).show() 
df_position.withColumn("ty",    regexp_extract(col("num"), r"-(\d+)", 1)).show() 

+--------------+-----------+-----------+
|           num|at_position|        mob|
+--------------+-----------+-----------+
|    1234567780|          0| 1234567780|
|    2234578996|          0| 2234578996|
|+1-12244567780|          3|12244567780|
|+32-2233567889|          4| 2233567889|
|+2-23456987312|          3|23456987312|
|+91-9087654123|          4| 9087654123|
|+23-9085761324|          4| 9085761324|
|+11-8091013345|          4| 8091013345|
+--------------+-----------+-----------+

+--------------+-----------+----+
|           num|at_position|code|
+--------------+-----------+----+
|    1234567780|          0|    |
|    2234578996|          0|    |
|+1-12244567780|          3|  +1|
|+32-2233567889|          4| +32|
|+2-23456987312|          3|  +2|
|+91-9087654123|          4| +91|
|+23-9085761324|          4| +23|
|+11-8091013345|          4| +11|
+--------------+-----------+----+

+--------------+-----------+-----------+
|           num|at_position|         ty|
+--------------+

In [0]:
from pyspark.sql.functions import expr

df_unpivot = df.select(
    "id",
    expr("""
        stack(3, 
              'math', math, #column,value
              'science', science,
              'english', english
        ) as (subject, marks)
    """)
)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

spark = SparkSession.builder.getOrCreate()

# Sample employee table
data = [
    (1, "Alice", None),
    (2, "Bob", 1),
    (3, "Charlie", 1),
    (4, "David", 2)
]
columns = ["emp_id", "name", "manager_id"]
employees_df = spark.createDataFrame(data, columns)

# Anchor member: start with Alice
anchor_df = employees_df.filter(employees_df.name == "Alice") \
                        .withColumn("level", lit(0))

hierarchy_df = anchor_df

while True:
    # Find next level employees
    next_level_df = employees_df.join(
        hierarchy_df,
        employees_df.manager_id == hierarchy_df.emp_id,
        "inner"
    ).select(
        employees_df.emp_id,
        employees_df.name,
        employees_df.manager_id,
        (hierarchy_df.level + 1).alias("level")
    )

    # Stop if no new rows
    if next_level_df.count() == 0:
        break

    hierarchy_df = hierarchy_df.union(next_level_df).distinct()

# Show hierarchy
hierarchy_df.orderBy("level", "emp_id").show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8880186937764656>, line 36
     24 next_level_df = employees_df.join(
     25     hierarchy_df,
     26     employees_df.manager_id == hierarchy_df.emp_id,
   (...)
     32     (hierarchy_df.level + 1).alias("level")
     33 )
     35 # Stop if no new rows
---> 36 if next_level_df.count() == 0:
     37     break
     39 hierarchy_df = hierarchy_df.union(next_level_df).distinct()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1923, in DataFrame._to_table(self)
   

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

# Main employees DataFrame
employees_data = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Carol"),
    (4, "David")
]
employees = spark.createDataFrame(employees_data, ["EmployeeID", "Name"])

# Another DataFrame containing IDs to filter
filter_data = [
    (1,),
    (3,)
]
filter_df = spark.createDataFrame(filter_data, ["EmployeeID"])

# Convert filter_df to a list (works if small)
id_list = [row["EmployeeID"] for row in filter_df.collect()]

# Use isin
filtered_employees = employees.filter(col("EmployeeID").isin(id_list))
filtered_employees.show()

+----------+-----+
|EmployeeID| Name|
+----------+-----+
|         1|Alice|
|         3|Carol|
+----------+-----+



In [0]:
print(filter_data.collect()[0][0])

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-8880186937764662>, line 1
----> 1 print(filter_data.collect()[0][0])

AttributeError: 'list' object has no attribute 'collect'

In [0]:
data = [(1, "Alice"), (2, "Bob")]
df = spark.createDataFrame(data, ["id", "name"])

rows = df.collect()[0]
print(rows["id"])
id_list = [row["id"] for row in df.collect()]
print(id_list)
from pyspark.sql.functions import max, min
rows = df.select(max("id").alias("max1"), min("name").alias("min1" )).collect()[0]

print(rows["min1"])

1
[1, 2]
Alice
